# VitalDB validation-only group retraining analysis

This CPU-compatible notebook validates and analyzes the 40 completed Colab runs. It does not train models and does not load held-out test data. Existing run directories are read-only inputs; all new tables, figures, and the report are written under `analysis/`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path

REPO_URL = 'https://github.com/khyitty/VitalDB-Feature-Selection.git'
REPO_DIR = Path('/content/VitalDB-Feature-Selection')
DRIVE_PROJECT_ROOT = Path('/content/drive/MyDrive/VitalDB-Feature-Selection')
DATASET_DIR = DRIVE_PROJECT_ROOT / 'data/modeling/full'
EXPERIMENT_DIR = DRIVE_PROJECT_ROOT / 'outputs/group_retraining_validation_only'
ANALYSIS_DIR = EXPERIMENT_DIR / 'analysis'
BOOTSTRAP_REPLICATES = 10_000
BOOTSTRAP_SEED = 20260715
print({'experiment': str(EXPERIMENT_DIR), 'analysis': str(ANALYSIS_DIR)})

## Repository and dependencies

A GPU is not required. The repository is updated with a fast-forward-only pull, and the existing Colab dependency guard preserves the installed PyTorch build.

In [ ]:
import os
import subprocess
import sys

if (REPO_DIR / '.git').is_dir():
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
CURRENT_COMMIT = subprocess.run(
    ['git', 'rev-parse', 'HEAD'], check=True, capture_output=True, text=True
).stdout.strip()
subprocess.run([
    sys.executable, 'scripts/install_colab_dependencies.py',
    '--requirements', 'requirements-colab.txt',
], check=True)
print('Analysis code commit:', CURRENT_COMMIT)

## Read-only input check and analysis

The analysis script blocks on missing, duplicate, incomplete, incompatible, non-CUDA, or test-evaluated runs. It also requires exact validation patient, timestamp, row, and target alignment before writing outputs.

In [ ]:
if not EXPERIMENT_DIR.is_dir():
    raise FileNotFoundError(f'Completed experiment directory is missing: {EXPERIMENT_DIR}')
for required_name in (
    'dataset_metadata.json',
    'preprocessing.pkl',
    'preprocessing_statistics.csv',
    'train_metadata.csv',
    'val_metadata.csv',
):
    if not (DATASET_DIR / required_name).is_file():
        raise FileNotFoundError(f'Missing modeling artifact: {required_name}')

subprocess.run([
    sys.executable, 'scripts/analyze_group_retraining.py',
    '--experiment-dir', str(EXPERIMENT_DIR),
    '--dataset-dir', str(DATASET_DIR),
    '--output-dir', str(ANALYSIS_DIR),
    '--bootstrap-replicates', str(BOOTSTRAP_REPLICATES),
    '--bootstrap-seed', str(BOOTSTRAP_SEED),
], check=True)

## Aggregate validation tables

In [ ]:
import pandas as pd
from IPython.display import display

condition_aggregate = pd.read_csv(ANALYSIS_DIR / 'validation_condition_aggregate.csv')
condition_contrasts = pd.read_csv(ANALYSIS_DIR / 'paired_condition_contrasts.csv')
model_contrasts = pd.read_csv(ANALYSIS_DIR / 'paired_model_contrasts.csv')
bootstrap = pd.read_csv(ANALYSIS_DIR / 'hierarchical_bootstrap_contrasts.csv')
pareto = pd.read_csv(ANALYSIS_DIR / 'pareto_candidates.csv')
display(condition_aggregate)
display(condition_contrasts)
display(model_contrasts)
display(bootstrap)
display(pareto)

## Validation-only figures

In [ ]:
from IPython.display import Image, display

figure_names = (
    'validation_seed_mae_distribution.png',
    'validation_paired_condition_lines.png',
    'validation_paired_delta_bootstrap_forest.png',
    'validation_feature_count_pareto.png',
    'validation_gru_attention_pairs.png',
)
for figure_name in figure_names:
    print(figure_name)
    display(Image(filename=str(ANALYSIS_DIR / 'figures' / figure_name)))

## Generated report excerpt

In [ ]:
report_path = ANALYSIS_DIR / 'validation_analysis_report.md'
report = report_path.read_text(encoding='utf-8')
print(report[:12_000])
print('Analysis outputs:', ANALYSIS_DIR)